## Notebook 8: CBA

In [8]:
import os
os.environ["CITY"] = "rome"   # pick the city here

In [9]:
# Generic bootstrap 
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
from cityheat.paths import make_P, ensure_out

# Choose city here
SLUG = globals().get("SLUG", os.environ.get("CITY", "rome")).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

→ City: rome  |  BASE=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome  OUT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome  INT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim


In [10]:
# city config + file paths from YAML 
cfg = C.get("cfg", {})                      # full YAML for the selected city
SLUG = cfg.get("slug", SLUG).lower()        
CITY = cfg.get("city_name", CITY)

paths    = cfg.get("files", {})             # {gvi_csv, lcz_candidates, cooling_coeffs_csv, ...}
osm_cfg  = cfg.get("osm", {})               # OSM settings used later in NB5
trees_cfg = cfg.get("trees", {})            # TARGET/CAP for NB5
urbclim   = cfg.get("urbclim", {})          # UrbClim folder/settings for NB5

lcz_candidates = [P(p) for p in paths.get("lcz_candidates", [])]
gvi_path = P(paths.get("gvi_csv", ""))

# FUA geopackage written in NB2 
fua_gpkg = Path(paths.get("fua_gpkg", f"{OUT}/{SLUG}_fua.gpkg"))

cool_csv = P(paths.get("cooling_coeffs_csv", ""))

# checks
print("SLUG/CITY:", SLUG, CITY)
print("GVI CSV:  ", gvi_path)
print("LCZ cand: ", [str(p) for p in lcz_candidates])
print("FUA GPKG: ", fua_gpkg)
print("Cooling CSV:", cool_csv)

SLUG/CITY: rome Rome
GVI CSV:   /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/gviRome/gvi_Rome.csv
LCZ cand:  ['/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_filter_v3.tif', '/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_v3.tif']
FUA GPKG:  /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/rome_fua.gpkg
Cooling CSV: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/CoolingEff/outer_2_wbgt_max.csv


**Loading from before**

In [11]:
# loading everything needed for the CBA 
from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT = Path(OUT)
INT = Path(INT)

TAB_DIR = OUT / "tables"

# Vegetation policy / ΔGVI 
trees_tbl = pd.read_csv(TAB_DIR / f"{SLUG}_trees_tbl.csv")

# diagnostics JSON with citywide ΔGVI (pop-weighted, points on 1–100 scale)
veg_diag_path = OUT / f"{SLUG}_veg_diagnostics.json"
veg_diag = json.loads(veg_diag_path.read_text())
citywide_dGVI_points_popw = veg_diag["citywide_dGVI_points_popw"]
print("Citywide pop-weighted ΔGVI (points, 1–100 scale):", citywide_dGVI_points_popw)

# AC coverage / municipal pop / energy use 
# coverage maps
ac_cov_npz = np.load(INT / f"ac_coverage_maps_{SLUG}.npz")
coverage_base   = ac_cov_npz["coverage_base"]
coverage_policy = ac_cov_npz["coverage_policy"]
CITY_MASK       = ac_cov_npz["CITY_MASK"].astype(bool)
HGT             = int(ac_cov_npz["HGT"])
WDT             = int(ac_cov_npz["WDT"])

# population on ref grid
pop_npz = np.load(INT / f"pop_on_ref_{SLUG}.npz")
pop_on_ref = pop_npz["pop"]

# municipio coverage table (pop_muni, ac_base_muni, ac_policy_muni)
muni_cov = pd.read_csv(OUT / f"muni_cov_{SLUG}.csv")

# AC consumption per municipality (kWh per AC user)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

# AC efficacy by age (for benefits, not directly cost-side) 
eff_buckets_path = INT / f"ac_eff_buckets_{SLUG}.json"
EFF_BUCKETS = json.loads(eff_buckets_path.read_text())
EFF_BUCKETS

Citywide pop-weighted ΔGVI (points, 1–100 scale): 2.3197


{'<15': 0.2, '15-64': 0.3, '65+': 0.4}

**Discount helpers**

In [12]:
import numpy as np

R = 0.03   # discount rate
T = 25     # time horizon

def pv_level_flow(annual, r=R, T=T):
    """PV of a constant annual amount paid in years 1..T."""
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1 + r) ** (-yrs)))

def pv_replacements(n_items, capex_per_item, r=R, T=T, life=20):
    """
    PV of buying 'n_items' at t=0 and replacing every 'life' years within horizon T.
    """
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return float(pv)

def annuity_factor(r=R, T=T):
    return (1 - (1 + r) ** (-T)) / r

AF = annuity_factor(R, T)
AF

17.413147691278027

**Trees: parametrisation of costs**

Some explanation on following code:
- Total index : how many GVI index points we need in total (once and for all) to reach policy target
- Investment rule: each new index point costs 10M once (investment), not every year. Spreading creation of those index points linearly over 25y : each year we create the same slice of change in GVI and pay 10M*that slice in that year. That's the change in GVI = sum(change in GVI/25) and 10M * change in GVI/25 per year.
- CAPEX PV: npv_capex_linear: takes ramp of yearly investments (same amount each year, over 25 years) and discounts them as flows in years 1...25. PV_trees_capex: NPV of the linear capex schedule
- IR vs EAC: TREES_CAPEX_T0: total undiscounted investment requirement
- Report EAC_capex_annuity=PV_trees_capex/AF : equivalent annual cost of our explicit linear ramp and EAC_capex_paper=TREES_CAPEX_T0/(1+R)^T*T) : as if all investments happens by T and we just annualise that discounted IR.
- O&M: npv_om_cohorts: treats O&M as yearly flows in years 1...25 each year we add a new cohort, and the number of active cohorts in year t is min(t, lifetime). We pay O&M per index point per year for all active cohorts and discount those flows. Matches description of O&M series that starts when trees are planted, accumulates cohorts, and is discounted. 

**Timing conventions for tree costs.**  
We consider a 25-year horizon and treat all cash flows as occurring at the end of each year (years 1–25). The total increase in the GVI index (ΔGVI, in 1–100 index points) implied by the tree policy is first converted into a **total investment requirement** using the rule that raising GVI by 1 index point costs 10 million euro once and for all. We then assume this investment is implemented along a **linear ramp**: the same fraction of ΔGVI is created in each year, and the corresponding investment is spread evenly over the 25 years. The functions `npv_capex_linear` and `npv_om_cohorts` compute the discounted present value of, respectively, this phased investment schedule and the recurrent O&M costs associated with overlapping planting cohorts. From these present values we derive equivalent annual costs (EACs) by dividing by the standard annuity factor. For comparison with the original study, we also report a “paper-style” EAC where the total undiscounted investment requirement is pushed to the end of the horizon, discounted once, and then divided by the number of years.

In [13]:
# Parameters from rule and regreen study 
CAPEX_PER_INDEX_PT = 10_000_000.0   # eur per 1 index point (1–100 scale)
CAPEX_PER_TREE     = 210.0          # eur per tree (REGREEN median)
OM_PER_TREE_YR     = 27.0           # eur per tree per year
LIFETIME_YEARS     = 25             # tree benefit/O&M lifetime 

# O&M per index point per year implied by tree-level numbers
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT
print("O&M per index point per year (EUR):", round(OM_PER_INDEX_PT_YR, 0))

# Total change in GVI in index point (1–100 SCALE) from trees_tbl 
# trees_tbl['dGVI'] is in 0–1 (fraction of max index); sum over Municipi, then ×100 => points
DELTA_INDEX = float(trees_tbl["dGVI"].clip(lower=0).sum()) * 100.0
print("Total ΔGVI index points (1–100 scale):", round(DELTA_INDEX, 2))

# For comparison: citywide pop-weighted ΔGVI (diagnostics)
print("Pop-weighted ΔGVI points (diagnostic):", citywide_dGVI_points_popw)

# CAPEX: linear ramp over T years 
def npv_capex_linear(delta_index_total, years=T, r=R,
                     capex_per_index=CAPEX_PER_INDEX_PT):
    """
    PV of a linear ramp: we add delta_index_total/years index points
    each year over 'years', and pay capex_per_index per point.
    All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years  # index points added per year
    pv = 0.0
    for t in range(1, years + 1):    # t = 1..years
        capex_t = capex_per_index * inc
        pv += capex_t / ((1 + r) ** t)
    return float(pv)

TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX   # undiscounted total “once and for all” cost
PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)

print(f"Trees — Total CAPEX requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")
print(f"Trees — NPV CAPEX (linear ramp):              €{PV_trees_capex:,.0f}")

# O&M: overlapping cohorts with constant per-index-point O&M 
def npv_om_cohorts(delta_index_total, years=T, r=R,
                   om_per_index_per_year=OM_PER_INDEX_PT_YR, lifetime=LIFETIME_YEARS):
    """
    O&M with overlapping cohorts: each year we add 'inc' index points
    and each cohort pays om_per_index_per_year * inc every year after planting,
    up to 'lifetime' years. All flows at the end of years 1..years.
    """
    inc = delta_index_total / years
    pv = 0.0
    for t in range(1, years + 1):    # t = 1..years
        active = min(t, lifetime)    # number of active cohorts in year t
        om_t = active * om_per_index_per_year * inc
        pv += om_t / ((1 + r) ** t)
    return float(pv)

PV_trees_om = npv_om_cohorts(DELTA_INDEX, years=T, r=R,
                             om_per_index_per_year=OM_PER_INDEX_PT_YR,
                             lifetime=LIFETIME_YEARS)

print(f"Trees — NPV O&M (cohorts):                   €{PV_trees_om:,.0f}")

# program totals 
PV_trees_total = PV_trees_capex + PV_trees_om
print(f"Trees — NPV total (CAPEX + O&M):             €{PV_trees_total:,.0f}")

# EACs: standard annuity vs paper-style formula 
AF = annuity_factor(R, T)

EAC_capex_annuity = PV_trees_capex / AF
EAC_om_annuity    = PV_trees_om    / AF
EAC_total_annuity = PV_trees_total / AF

# Article-like “investment requirement divided by (1+r)^T * T”
EAC_capex_paper = TREES_CAPEX_T0 / ((1 + R)**T * T)

print(f"Trees — EAC CAPEX (annuity):   €{EAC_capex_annuity:,.0f}/yr")
print(f"Trees — EAC O&M (annuity):     €{EAC_om_annuity:,.0f}/yr")
print(f"Trees — EAC total (annuity):   €{EAC_total_annuity:,.0f}/yr")
print(f"Trees — EAC CAPEX (paper-style): €{EAC_capex_paper:,.0f}/yr")

O&M per index point per year (EUR): 1285714.0
Total ΔGVI index points (1–100 scale): 30.23
Pop-weighted ΔGVI points (diagnostic): 2.3197
Trees — Total CAPEX requirement (undiscounted): €302,333,765
Trees — NPV CAPEX (linear ramp):              €210,583,300
Trees — NPV O&M (cohorts):                   €310,733,610
Trees — NPV total (CAPEX + O&M):             €521,316,910
Trees — EAC CAPEX (annuity):   €12,093,351/yr
Trees — EAC O&M (annuity):     €17,844,770/yr
Trees — EAC total (annuity):   €29,938,120/yr
Trees — EAC CAPEX (paper-style): €5,775,852/yr


- DELTA_INDEX: citywide change in GVI on 1-100 scale
- inc = DELTA_INDEX/years is change in GVI / 25 each year => linear path
- capex_t = 10Meur * inc is the 10Meur*change in GVI/25 rule
- Discounting stream year by year to get PV_trees_capex
- TREES_CAPEX_T0: paper way of doing it: what if we did everything upfront

- each year we plant inc index points (new cohort)
- each cohort costs om_per_index_per_year * inc every year
- after t years, there are t+1 overlapping cohorts (until we cap at lifetime)
- om_t is exactly sum over all active cohorts
- then we discount om_t year by year
- here, we choose that all cohorts have same per-index-point O&M each year for lifetime years

BIG QUESTION HERE BECAUSE I NEVER UNDERSTAND:
About the annualisation:
- We have two different annualisation ideas in the code
- 1. Economically consistent one with our ramp (PV_trees_capex, AF, EAC_capex_annuity)
     Interpretation: npv_capex_linear gives the present value of the phased CAPEX stream
     Dividing by the annuity factor (that only depends on T and r), converts that pV into
     a constant equivalent annual cost over 25 years.
     Basically, given a specific investment path (here: linear ramp), we discount it,
     then convert to PV into a flat annual amount.
  2. EAC_capex_paper: this is not the same thing as annuity based on ramp. It's like the
     paper of Giacomo: take undiscounted total investment requirement IR, push it to year
     T, discount it once, divide by T. It's to get the "average discounted annual cost"
     but assumes everything at the end and does not reflect our explicit tamp path. The
     correct one should be the 1. but need to ask Giacomo more about this. 

More explanation about this...
- the annuity EAC: takes the actual NPV of the ramped CAPEX (and O&M) streams, divide by the standard annuity factor. "Constant yearly cost, over 25 years, that is financially equivalent to this time varying cashflow"
- Paper : take the total undiscounted investment requirement (as if all invested once), shift it to year T then divide by T. It doesn't reflect our ramp. It's a shortcut to get an average discounted annual cost from a single IR number. We use it for comparability.

From what I understand, what I do is what we discussed: we have a total increase in GVI needed (delta_index), we assume a linear path over 25 years and each year we add change in GVI/25 index points. We use the rule: 1 index points: 10M once, not every year. So each year we invest 10M*(deltaGVI/25). We discount that year by year to get an NPV of capex: npv_capex_linear. For O&M, cohort logic: each year, new cohort planted, each cohort pays the same O&M per index point each year, in year t we have mint(, lifetime) active cohorts, we discount the whole panel of O&M flows: npv_om_cohorts. We do the standard annuity step : taking NPV of CAPEX (or total CAPEX + O&M) and dividing by annuity factor sum from t=1 to T of (1+r)^t to get a constant yearly cost that is financially equivalent to the detailed schedule. 

- npv_capex_linear: discounted sum of future cashflows with a linear ramp
- npv_om_cohorts: O&M cohorts idea, discounted year by year
- EAC by dividing by annuity factor. Depends only on r and T because it’s the present value of paying “1€ every year” over T years. It doesn’t have to know about how cohorts build up; that’s already into the NPV. 

What is happening in the paper? IRc is a total undiscounted investment requirement (already aggregated over the 25 y horizon). Then, you push that entire amount to year T, discount it once by (1+r)^T and divide by T. It's: taking the total investment we would need in today's euros, pretending it's all paid in year T, discounting that once, then just spreading the discounted lump evenly over T years. 

In the paper, the formula gave an “average discounted yearly investment” directly from a total IR, without specifying a time pattern.

Here, we now do specify the time pattern explicitly (linear ramp of ΔGVI and overlapping O&M cohorts). So I compute NPV as the discounted sum of those yearly cashflows, and then convert that NPV into a constant equivalent annual cost using the standard annuity factor for r = 3% and T=25.

This annuity-based EAC is the one that’s fully consistent with the dynamic ramp and with the way we treat benefits (yearly avoided deaths). I still report the paper-style EAC (IR/(1+r)^T/T) alongside, to keep direct comparability with your article.

**Sensitivity paved streets**

TO DO IF IT'S RIGHT, WITH 5379.0 instead of 210.

**Cost AC**

In [14]:
# AC PARAMS (could also be read from cfg["ac"]) 
AC_CAPEX_PER_USER   = 500.0   # € per AC unit
AC_MAINT_RATE       = 0.05    # fraction of CAPEX per year
AC_LIFETIME_YEARS   = 10      # replacement cycle
TARIFF_EUR_PER_KWH  = 0.25    # €/kWh

# merge coverage by municipio with kWh per AC user by municipio
df = (muni_cov.merge(muni_tbl_all[["muni_id", "kwh_per_user_muni"]],
                     on="muni_id", how="left")
             .fillna({"kwh_per_user_muni": 0.0}))

# only inside Municipi
df = df.loc[df["muni_id"] > 0].copy()

# increase in AC share per municipio (clip negative)
df["dshare"] = (df["ac_policy_muni"] - df["ac_base_muni"]).clip(lower=0.0)

# total new AC users from the policy
added_users = float(np.nansum(df["pop_muni"] * df["dshare"]))
print(f"AC — added users ≈ {added_users:,.0f}")

# per-user annual maintenance
maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER

# per-year citywide electricity cost due to new users
elec_eur_per_yr = float(np.nansum(
    df["pop_muni"] * df["dshare"] * df["kwh_per_user_muni"]
)) * TARIFF_EUR_PER_KWH

print(f"AC — annual electricity € (undiscounted): {elec_eur_per_yr:,.0f}")

# DISCOUNT TO PV 
PV_ac_capex = pv_replacements(
    n_items=added_users,
    capex_per_item=AC_CAPEX_PER_USER,
    r=R, T=T, life=AC_LIFETIME_YEARS
)

PV_ac_maint = pv_level_flow(added_users * maint_per_user_yr, r=R, T=T)
PV_ac_elec  = pv_level_flow(elec_eur_per_yr,                r=R, T=T)

PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec

print(f"AC — PV capex €{PV_ac_capex:,.0f}")
print(f"AC — PV maint €{PV_ac_maint:,.0f}")
print(f"AC — PV elec  €{PV_ac_elec:,.0f}")
print(f"AC — PV total €{PV_ac_total:,.0f}")

EAC_ac_total = PV_ac_total / AF
print(f"AC — EAC total (annuity): €{EAC_ac_total:,.0f}/yr")

AC — added users ≈ 189,806
AC — annual electricity € (undiscounted): 30,752,015
AC — PV capex €218,065,388
AC — PV maint €82,628,056
AC — PV elec  €535,489,386
AC — PV total €836,182,829
AC — EAC total (annuity): €48,020,200/yr


**Benefits and summary**

- Using the three CLIMADA points (2030, 2040, 2050)

- Interpolating them into a smooth yearly series over 25 years

- Applying the tree ramp only to the tree component

- Discounting the full yearly streams

In [15]:
# Benefits and summary

HORIZON_YEARS   = T
DISCOUNT_RATE   = R
TREE_RAMP_YEARS = 12

# smooth ramp 0 to 1 over TREE_RAMP_YEARS
tree_ramp = np.minimum(np.arange(1, HORIZON_YEARS+1) / TREE_RAMP_YEARS, 1.0)

def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows) + 1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1 + r) ** (-yrs)))

import numpy as np
import pandas as pd
from pathlib import Path

INT = Path(INT)

# Trees (avoided deaths per year, trees vs current-AC baseline)
avo_trees = pd.read_csv(
    INT / f"annual_heat_deaths_avoided_trees_curr_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

# AC (policy vs baseline, overall avoided deaths per year) – Series indexed by year
avo_ac = pd.read_csv(
    INT / f"annual_heat_deaths_climada_avoided_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

# Trees + AC policy (vs current AC baseline) 
avo_both = pd.read_csv(
    INT / f"annual_heat_deaths_avoided_treesplusAC_curr_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

print("Trees – avoided deaths per year:")
print(avo_trees, "\n")

print("AC – avoided deaths per year:")
print(avo_ac, "\n")

print("Both (trees+AC vs current AC) – avoided deaths per year:")
print(avo_both)

# Build yearly benefit paths by interpolation between 2030/2040/2050
START_YEAR = int(min(avo_trees.index))
YEARS = np.arange(START_YEAR, START_YEAR + HORIZON_YEARS, dtype=int)

def interpolate_to_horizon(s, years):
    s = s.sort_index()
    known = s.index.to_numpy(int)
    vals  = s.to_numpy(float)
    out = np.interp(years, known, vals)
    out[years <= known[0]]  = vals[0]
    out[years >= known[-1]] = vals[-1]
    return out

# interpolate each scenario
trees_full = interpolate_to_horizon(avo_trees, YEARS)
ac_full    = interpolate_to_horizon(avo_ac,    YEARS)
both_full  = interpolate_to_horizon(avo_both,  YEARS)

# ramp
tree_ramp = np.minimum(np.arange(1, HORIZON_YEARS+1) / TREE_RAMP_YEARS, 1.0)

# Trees-only: ramp the full trees effect vs current AC
trees_yr        = trees_full
trees_yr_ramped = trees_yr * tree_ramp

# AC-only: no ramp
ac_yr = ac_full

# BOTH: AC immediate + tree effect ramp, but now the tree effect is conditional on AC
trees_cond_yr   = both_full - ac_full   # extra effect of trees when AC policy is present
both_yr_ramped  = ac_yr + trees_cond_yr * tree_ramp

# PV of avoided deaths
PV_b_tree = pv_of_stream(trees_yr_ramped, DISCOUNT_RATE)
PV_b_ac   = pv_of_stream(ac_yr,           DISCOUNT_RATE)
PV_b_both = pv_of_stream(both_yr_ramped,  DISCOUNT_RATE)

print("PV benefits (avoided deaths) —",
      f"Trees: {PV_b_tree:,.2f} | AC: {PV_b_ac:,.2f} | Both: {PV_b_both:,.2f}")

# sensitivity: undiscounted cumulative avoided deaths (no time weighting) 

CUM_b_tree = float(np.sum(trees_yr_ramped))   # trees: ramped benefits
CUM_b_ac   = float(np.sum(ac_yr))            # AC: no ramp (benefits immediate)
CUM_b_both = float(np.sum(both_yr_ramped))   # both: AC + ramped tree extra

print("Cumulative avoided deaths over horizon (undiscounted) — "
      f"Trees: {CUM_b_tree:,.2f} | AC: {CUM_b_ac:,.2f} | Both: {CUM_b_both:,.2f}")

# incremental trees benefit given AC policy
PV_b_tree_cond   = PV_b_both - PV_b_ac      # PV avoided deaths, trees on top of AC
CUM_b_tree_cond  = CUM_b_both - CUM_b_ac    # cumulative (undiscounted), trees on top of AC

Trees – avoided deaths per year:
year
2030    8.789494
2040    8.467261
2050    8.996251
Name: overall, dtype: float64 

AC – avoided deaths per year:
year
2030    26.123653
2040    24.426194
2050    26.856089
Name: overall, dtype: float64 

Both (trees+AC vs current AC) – avoided deaths per year:
year
2030    34.385183
2040    32.385640
2050    35.311887
Name: overall, dtype: float64
PV benefits (avoided deaths) — Trees: 109.62 | AC: 446.38 | Both: 549.42
Cumulative avoided deaths over horizon (undiscounted) — Trees: 170.73 | AC: 643.07 | Both: 803.55


In [16]:
from pathlib import Path
import pandas as pd

# Baseline heat deaths with current AC (RAW CLIMADA, no scaling)
base_ac_path = Path(INT) / f"annual_heat_deaths_curr_AC_base_{SLUG}.csv"

tmp = pd.read_csv(base_ac_path)
print("Columns in current-AC baseline CSV:", list(tmp.columns))

# infer year index
if "year" in tmp.columns:
    baseline_by_age = tmp.set_index("year")
else:
    baseline_by_age = tmp.set_index(tmp.columns[0])
    baseline_by_age.index.name = "year"

display(baseline_by_age)

# total baseline deaths
if set(["<15", "15-64", "65+"]).issubset(baseline_by_age.columns):
    baseline_total = baseline_by_age[["15-64", "65+", "<15"]].sum(axis=1)
else:
    baseline_total = baseline_by_age["overall"]

baseline_total.name = "baseline_total"

print("Baseline total heat-attributable deaths per year (current AC, RAW):")
display(baseline_total)

# align avoided deaths with baseline years
avo_trees_al = avo_trees.reindex(baseline_total.index)
avo_ac_al    = avo_ac.reindex(baseline_total.index)
avo_both_al  = avo_both.reindex(baseline_total.index)

benefit_pct = pd.DataFrame({
    "baseline_total": baseline_total,
    "avo_trees":      avo_trees_al,
    "avo_ac":         avo_ac_al,
    "avo_both":       avo_both_al,
})

benefit_pct["trees_pct"] = 100 * benefit_pct["avo_trees"] / benefit_pct["baseline_total"]
benefit_pct["ac_pct"]    = 100 * benefit_pct["avo_ac"]    / benefit_pct["baseline_total"]
benefit_pct["both_pct"]  = 100 * benefit_pct["avo_both"]  / benefit_pct["baseline_total"]
benefit_pct["trees_on_top_pct"] = benefit_pct["both_pct"] - benefit_pct["ac_pct"]

benefit_pct.round(2)

Columns in current-AC baseline CSV: ['year', '<15', '15-64', '65+', 'overall']


,<15,15-64,65+,overall
year,,,,
2030,3.382917,77.091174,590.414426,670.888517
2040,3.276677,73.214870,551.934678,628.426226
2050,3.452808,78.970665,607.082165,689.505638


Baseline total heat-attributable deaths per year (current AC, RAW):


year
2030    670.888517
2040    628.426226
2050    689.505638
Name: baseline_total, dtype: float64

,baseline_total,avo_trees,avo_ac,avo_both,trees_pct,ac_pct,both_pct,trees_on_top_pct
year,,,,,,,,
2030,670.89,8.79,26.12,34.39,1.31,3.89,5.13,1.23
2040,628.43,8.47,24.43,32.39,1.35,3.89,5.15,1.27
2050,689.51,9.00,26.86,35.31,1.30,3.89,5.12,1.23


In [17]:
def safe_ratio(c, b):
    return float(c / b) if (b is not None and b > 1e-9) else np.inf

summary_main = pd.DataFrame([
    {
        "Policy": "Trees only (vs current AC)",
        "PV_cost_eur": PV_trees_total,
        "PV_avoided_deaths": PV_b_tree,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, PV_b_tree),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": 0.0,
    },
    {
        "Policy": "AC only (vs current AC)",
        "PV_cost_eur": PV_ac_total,
        "PV_avoided_deaths": PV_b_ac,
        "Cost_per_avoided_death_eur": safe_ratio(PV_ac_total, PV_b_ac),
        "EAC_capex_annuity_eur_per_yr": 0.0,
        "EAC_om_annuity_eur_per_yr":    0.0,
        "EAC_total_annuity_eur_per_yr": EAC_ac_total,
        "EAC_capex_paper_eur_per_yr":   np.nan,
        "added_AC_users": added_users,
    },
    {
        "Policy": "Both (trees+AC vs current AC)",
        "PV_cost_eur": PV_trees_total + PV_ac_total,
        "PV_avoided_deaths": PV_b_both,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total + PV_ac_total, PV_b_both),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity + EAC_ac_total,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": added_users,
    },
    {
        # marginal trees effect if AC policy is already implemented
        "Policy": "Trees (incremental, on top of AC policy)",
        "PV_cost_eur": PV_trees_total,
        "PV_avoided_deaths": PV_b_tree_cond,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, PV_b_tree_cond),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": 0.0,
    },
]).round(2)

summary_main

,Policy,PV_cost_eur,PV_avoided_deaths,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Trees only (vs current AC),5.213169e+08,109.62,4755802.70,12093350.58,17844769.69,29938120.28,5775851.59,0.00
1,AC only (vs current AC),8.361828e+08,446.38,1873254.91,0.00,0.00,48020199.67,NaN,189806.13
2,Both (trees+AC vs current AC),1.357500e+09,549.42,2470803.00,12093350.58,17844769.69,77958319.95,5775851.59,189806.13
3,"Trees (incremental, on top of AC policy)",5.213169e+08,103.04,5059522.01,12093350.58,17844769.69,29938120.28,5775851.59,0.00


**On a PV budget**

**Sensitivity**